<a href="https://colab.research.google.com/github/AdinaGo/AI-model-agent/blob/main/agent/agent_logic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## שלב 4 – בניית סוכן הבינה המלאכותית

### טעינת המודל המאומן לצורך הפעלת הסוכן

בשלב זה נטען מודל ה־DistilBERT שאומן ונשמר קודם לכן במהלך תהליך ה־Fine-Tuning. מטרת שלב זה היא לאפשר לסוכן הבינה המלאכותית להשתמש במודל המאומן לצורך סיווג פניות חדשות של לקוחות.

תחילה נטענים ה־Tokenizer והמודל מתוך התיקייה שבה נשמרה הגרסה הטובה ביותר של המודל. ה־Tokenizer אחראי להמיר טקסטים לפורמט המספרי הנדרש על ידי המודל, בעוד שהמודל עצמו משמש לביצוע תחזיות על פניות חדשות.

לאחר מכן נקבעת סביבת החישוב המתאימה (CPU או GPU), והמודל מועבר אליה לצורך ביצוע חישובים בצורה יעילה. לבסוף, המודל מועבר למצב הערכה (`eval()`), אשר מבטל מנגנונים המשמשים במהלך האימון בלבד, כגון Dropout, ומבטיח שהתחזיות יהיו יציבות ועקביות.

שלב זה מסמן את המעבר מתהליך פיתוח ואימון המודל לשלב השימוש המעשי בו. מעתה הסוכן משתמש במודל המאומן לצורך זיהוי נושא הפנייה והפעלת תהליך התמיכה האוטומטי.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

saved_model_path = "/content/drive/MyDrive/banking77_model"
agent_tokenizer = AutoTokenizer.from_pretrained(saved_model_path)
agent_model = AutoModelForSequenceClassification.from_pretrained(saved_model_path)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [ ]:
agent_device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
agent_model.to(agent_device)
agent_model.eval()

print("Saved DistilBERT model loaded successfully for the Agent.")

Saved DistilBERT model loaded successfully for the Agent.


### סיווג פנייה חדשה באמצעות DistilBERT

פונקציה זו אחראית על סיווג אוטומטי של פניות חדשות המתקבלות במערכת באמצעות מודל ה־DistilBERT שאומן במהלך הפרויקט.

כאשר מתקבלת פנייה חדשה, הטקסט מועבר תחילה ל־Tokenizer של המודל. בשלב זה הטקסט מומר לייצוג מספרי (Tokens) שהמודל מסוגל לעבד. בנוסף, מבוצעות פעולות כגון חיתוך טקסטים ארוכים (Truncation), השלמת Padding והתאמת אורך הקלט לפורמט שבו המודל אומן.

לאחר מכן הנתונים מועברים לסביבת החישוב המתאימה (CPU או GPU), והמודל מבצע חיזוי ללא חישוב גרדיאנטים (`torch.no_grad()`), מכיוון שבשלב זה מתבצעת הסקה (Inference) בלבד ולא אימון.

המודל מחזיר ציון (Logits) עבור כל אחת מהקטגוריות האפשריות. הקטגוריה בעלת הציון הגבוה ביותר נבחרת כתחזית הסופית באמצעות פונקציית `argmax`.

לבסוף, מזהה הקטגוריה המספרי מומר בחזרה לשם הקטגוריה המקורי באמצעות מיפוי התוויות שנשמר יחד עם המודל, והפונקציה מחזירה את נושא הפנייה שזוהה.

פונקציה זו מהווה את רכיב הסיווג המרכזי של הסוכן, והיא אחראית על זיהוי אוטומטי של סוג הפנייה לפני מעבר לשלבי קבלת ההחלטות ויצירת התגובה.


In [ ]:
def classify_inquiry(text):
    inputs = agent_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    inputs = {key: value.to(agent_device) for key, value in inputs.items()}

    with torch.no_grad():
        outputs = agent_model(**inputs)

    predicted_id = torch.argmax(outputs.logits, dim=1).item()

    predicted_topic = agent_model.config.id2label[predicted_id]

    return predicted_topic

### מנגנון שליפת מדיניות הטיפול

לאחר שהמודל מזהה את נושא הפנייה, השלב הבא בתהליך הוא קביעת אופן הטיפול המתאים. לצורך כך נבנתה פונקציה בשם `lookup_policy`, אשר משמשת כמאגר ידע עסקי (Knowledge Base) עבור הסוכן.

הפונקציה מכילה אוסף של כללי טיפול המוגדרים מראש עבור כל אחת מהקטגוריות האפשריות במערכת. לכל קטגוריה משויכת מדיניות טיפול המתארת את הפעולות, הבדיקות או ההנחיות שיש לבצע על מנת לטפל בפניית הלקוח בצורה נכונה.

כאשר מתקבלת קטגוריה ממודל הסיווג, הפונקציה מאתרת את מדיניות הטיפול המתאימה ומחזירה אותה לשימוש בשלבים הבאים של הסוכן. במקרה שבו לא נמצאה מדיניות ייעודית עבור הקטגוריה שהתקבלה, מוחזרת מדיניות כללית המשמשת כברירת מחדל.

רכיב זה מדגים כיצד ניתן לשלב בין יכולות למידת מכונה לבין ידע עסקי ארגוני. בעוד שמודל הבינה המלאכותית אחראי על הבנת תוכן הפנייה וזיהוי הנושא שלה, מנגנון המדיניות אחראי על קבלת ההחלטה העסקית ועל הנחיית תהליך הטיפול המתאים.

גישה זו מאפשרת להפריד בין רכיב הסיווג לבין הלוגיקה העסקית של הארגון, ובכך ליצור מערכת גמישה וקלה יותר לתחזוקה ולעדכון.


In [ ]:
def lookup_policy(topic):

    policies = {
        "cards": "For card-related issues, verify the card status, delivery status, and activation details.",
        "payments": "For payment issues, check the transaction status. Duplicate charges should be reviewed and refunded if confirmed.",
        "transfers": "For transfer issues, check the transfer status, recipient details, and expected processing time.",
        "exchange": "For exchange-related issues, explain the current exchange rate, supported currencies, and applicable fees.",
        "cash_atm": "For ATM or cash withdrawal issues, check the ATM transaction logs. Investigation may take several business days.",
        "account": "For account-related issues, verify the customer's account details and update personal information if required.",
        "security": "Security-related issues should be escalated immediately. Identity verification must be completed before further action."
    }

    return policies.get(
        topic,
        "This inquiry should be handled according to the general customer support policy."
    )

### מימוש סוכן התמיכה האוטומטי

בשלב הסופי של הפרויקט פותח סוכן תמיכה אוטומטי המשלב את כל הרכיבים שנבנו לאורך העבודה לכדי מערכת אחת הפועלת מקצה לקצה.

כאשר מתקבלת פנייה חדשה מלקוח, הסוכן מפעיל תחילה את מודל ה־DistilBERT המאומן על מנת לזהות את נושא הפנייה ולסווג אותה לאחת מהקטגוריות שהוגדרו בפרויקט. לאחר זיהוי הקטגוריה המתאימה, הסוכן שולף את מדיניות הטיפול הרלוונטית מתוך מאגר הידע העסקי באמצעות פונקציית `lookup_policy`.

בשלב הבא המערכת משלבת את המידע שהתקבל ממודל הסיווג וממדיניות הטיפול, ויוצרת באופן אוטומטי טיוטת תגובה מסודרת ללקוח. התגובה כוללת את נושא הפנייה שזוהה, את הנחיות הטיפול המתאימות ואת נוסח התגובה המוצע.

פונקציה זו מהווה את מנגנון התיאום המרכזי של הסוכן, מכיוון שהיא מחברת בין רכיב הבינה המלאכותית, רכיב קבלת ההחלטות העסקי ורכיב יצירת התגובה. כתוצאה מכך מתקבל תהליך עבודה מלא המתחיל בפניית לקוח בשפה טבעית ומסתיים בהפקת תגובה מותאמת באופן אוטומטי.

רכיב זה מדגים כיצד ניתן לשלב מודל למידת מכונה עם כללים עסקיים כדי לבנות מערכת תמיכה חכמה, המסוגלת לייעל את תהליך הטיפול בפניות ולספק מענה אחיד ומהיר ללקוחות.


In [ ]:
def customer_support_agent(message):

    topic = classify_inquiry(message)

    policy = lookup_policy(topic)

    response = f"""
Detected category: {topic}

Relevant company policy:
{policy}

Draft response:

Dear Customer,

Thank you for contacting us.

We reviewed your inquiry and identified it as related to {topic}.

According to our support policy:

{policy}

We appreciate your patience and will continue assisting you until the issue is resolved.

Best regards,
Customer Support Team
"""

    return response

### הדגמת הסוכן על פניות לדוגמה

לאחר השלמת פיתוח הסוכן, בוצעה סדרת בדיקות על מספר פניות לדוגמה המייצגות סוגים שונים של בקשות ובעיות נפוצות בתחום הבנקאות.

מטרת הבדיקה הייתה לוודא שכל רכיבי המערכת פועלים יחד בצורה תקינה כחלק מתהליך עבודה מלא. עבור כל פנייה, הסוכן ביצע תחילה סיווג אוטומטי באמצעות מודל ה־DistilBERT, לאחר מכן שלף את מדיניות הטיפול המתאימה מתוך מאגר הידע העסקי, ולבסוף יצר טיוטת תגובה ללקוח.

הפניות שנבדקו כללו מגוון נושאים כגון תשלומים, כרטיסים, עדכון פרטי חשבון, אימות זהות והעברות כספים. תוצאות הבדיקה הראו כי הסוכן הצליח לזהות את נושא הפנייה בצורה נכונה, להתאים את מדיניות הטיפול הרלוונטית ולייצר תגובה עקבית ומסודרת בהתאם לנהלי הארגון.

בדיקות אלו מדגימות את יכולתה של המערכת לבצע תהליך אוטומטי מלא – החל מקבלת פנייה בשפה טבעית, דרך הבנת תוכנה וקבלת החלטה עסקית, ועד ליצירת תגובה מותאמת ללקוח.


In [ ]:
test_messages = [
    "I was charged twice for my card payment.",
    "My card has not arrived yet.",
    "I need to change my personal details.",
    "I need help with identity verification.",
    "My transfer is still pending."
]

for message in test_messages:
    print("=" * 80)
    print("Customer message:", message)
    print(customer_support_agent(message))

Customer message: I was charged twice for my card payment.

Detected category: payments

Relevant company policy:
For payment issues, check the transaction status. Duplicate charges should be reviewed and refunded if confirmed.

Draft response:

Dear Customer,

Thank you for contacting us.

We reviewed your inquiry and identified it as related to payments.

According to our support policy:

For payment issues, check the transaction status. Duplicate charges should be reviewed and refunded if confirmed.

We appreciate your patience and will continue assisting you until the issue is resolved.

Best regards,
Customer Support Team

Customer message: My card has not arrived yet.

Detected category: cards

Relevant company policy:
For card-related issues, verify the card status, delivery status, and activation details.

Draft response:

Dear Customer,

Thank you for contacting us.

We reviewed your inquiry and identified it as related to cards.

According to our support policy:

For card-rel

### תוצאות הדגמת הסוכן

סוכן הבינה המלאכותית השלים בהצלחה את כל שלבי תהליך העבודה.

עבור כל פנייה שהוזנה למערכת, הסוכן:

1. זיהה את נושא הפנייה באמצעות מודל הסיווג.
2. שלף את מדיניות הטיפול המתאימה מתוך מאגר הידע העסקי.
3. יצר טיוטת תגובה מקצועית ומסודרת עבור הלקוח.

דוגמאות הבדיקה הראו כי המערכת הצליחה לסווג בצורה נכונה פניות בנושאי תשלומים, כרטיסים, ניהול חשבון, אבטחה והעברות כספים, ולהתאים לכל אחת מהן את מדיניות הטיפול הרלוונטית.

תוצאות אלו מדגימות את יכולתה של המערכת לשלב בין סיווג אוטומטי המבוסס על למידת מכונה, שליפת מדיניות עסקית ויצירת תגובות אוטומטיות, כחלק מתהליך תמיכה חכם מקצה לקצה.


## סיכום הפרויקט

בפרויקט זה פותחה מערכת מבוססת בינה מלאכותית לסיווג אוטומטי של פניות לקוחות בתחום הבנקאות.

במהלך העבודה בוצעו שלבי ניתוח נתונים, עיבוד טקסט, בניית מודל בסיס מסוג SVM, ביצוע Fine-Tuning למודל DistilBERT ופיתוח סוכן תמיכה אוטומטי.

תוצאות הניסוי הראו כי מודל DistilBERT השיג את הביצועים הטובים ביותר, עם Accuracy ו־Weighted F1-Score של כ־97.6%.

לבסוף פותח סוכן תמיכה אוטומטי המשלב סיווג פניות, שליפת מדיניות עסקית ויצירת טיוטת תגובה ללקוח. המערכת מדגימה כיצד ניתן לשלב למידת מכונה, עיבוד שפה טבעית וכללים עסקיים בתהליך עבודה אוטומטי מקצה לקצה.

הפרויקט ממחיש את הפוטנציאל של פתרונות מבוססי בינה מלאכותית לשיפור תהליכי שירות לקוחות, לקיצור זמני טיפול וליצירת חוויית שירות אחידה ויעילה יותר.
